In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

results_dir = Path("evaluation/results/table5")
if not results_dir.exists():
    raise FileNotFoundError(
        f"{results_dir} not found. Run `bash scripts/reproduce_table5.sh` first."
    )

## Headline numbers

First a quick textual look at the four headline All-acc values — the paper headline of 0.868 LR-Routing should appear here.

In [ ]:
def read_tsv(path):
    with path.open() as f:
        header = f.readline().strip().split("\t")
        rows = [dict(zip(header, line.strip().split("\t"))) for line in f if line.strip()]
    return rows

comparison = read_tsv(results_dir / "comparison_results.tsv")
print(f"{'Method':<14} {'All_acc':>8} {'SER_MAE':>8}")
print("-" * 32)
for row in comparison:
    print(f"{row['method']:<14} {float(row['All_acc']):>8.4f} {float(row['SER_MAE']):>8.4f}")

## Per-example routing decisions

How many test examples does each routing strategy send to LoRA vs NLI? `routing_details.json` records both the score-threshold choice and the LR-router choice for every test example. We also have the oracle choice (the per-example label used as the LR-router's training target) so we can see how close each method comes to optimal routing.

In [ ]:
with (results_dir / "routing_details.json").open() as f:
    details = json.load(f)

examples = details["examples"]
score_choices = Counter(ex["score_routing_choice"] for ex in examples)
lr_choices = Counter(ex["lr_routing_choice"] for ex in examples)
oracle_choices = Counter(ex["oracle_label"] for ex in examples)

labels = ["NLI (0)", "LoRA (1)"]
data = {
    "ScoreRouting": [score_choices.get(0, 0), score_choices.get(1, 0)],
    "LR-Routing":   [lr_choices.get(0, 0), lr_choices.get(1, 0)],
    "Oracle":       [oracle_choices.get(0, 0), oracle_choices.get(1, 0)],
}

fig, ax = plt.subplots(figsize=(7, 4))
x_pos = list(range(len(labels)))
width = 0.25
for i, (name, counts) in enumerate(data.items()):
    offsets = [p + (i - 1) * width for p in x_pos]
    ax.bar(offsets, counts, width, label=name)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel("# test examples")
ax.set_title("Per-example routing: which method each strategy picks")
ax.legend()
plt.tight_layout()
plt.show()

## Score-threshold sweep

How does All-acc on the dev split move as we sweep the ranker top-score threshold? The threshold the script ends up picking maximises dev All-acc.

In [ ]:
sweep = read_tsv(results_dir / "score_routing_sweep.tsv")
thresholds = [float(r["threshold"]) for r in sweep]
all_acc = [float(r["All_acc"]) for r in sweep]
n_lora = [int(r["n_lora"]) for r in sweep]
n_nli = [int(r["n_nli"]) for r in sweep]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(thresholds, all_acc, marker="o")
ax1.set_xlabel("Score threshold")
ax1.set_ylabel("All-acc (dev)")
ax1.set_title("Score-threshold sweep — All-acc vs threshold")
ax1.grid(alpha=0.3)

ax2.plot(thresholds, n_lora, marker="o", label="# routed to LoRA")
ax2.plot(thresholds, n_nli, marker="s", label="# routed to NLI")
ax2.set_xlabel("Score threshold")
ax2.set_ylabel("# dev examples")
ax2.set_title("How threshold shifts the routing balance")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## LR feature importance

The LR routing model learns coefficients over 22 per-example features (3 score features + 3 MR complexity features + 3 NLI features + 13 topic one-hots). The largest-magnitude coefficients tell us what signals the router is leaning on.

In [ ]:
imp = read_tsv(results_dir / "lr_feature_importance.tsv")
# Sort by |coefficient|, take top 12
imp_sorted = sorted(imp, key=lambda r: abs(float(r["coefficient"])), reverse=True)[:12]
feat_names = [r["feature"] for r in imp_sorted]
coefs = [float(r["coefficient"]) for r in imp_sorted]
colors = ["#4c72b0" if c > 0 else "#c44e52" for c in coefs]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feat_names[::-1], coefs[::-1], color=colors[::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("LR coefficient (positive = favours routing to LoRA)")
ax.set_title("Top 12 LR routing features by |coefficient|")
plt.tight_layout()
plt.show()

## Per-topic routing pattern

Across the 12 evaluation topics, do the LR and score-threshold routers agree? Topics where the two diverge are where the LR's additional features (NLI coverage, MR complexity) are pulling its decision away from the simple top-score threshold.

In [ ]:
from collections import defaultdict

topic_lr = defaultdict(lambda: [0, 0])
topic_score = defaultdict(lambda: [0, 0])
for ex in examples:
    t = ex.get("topic", "unknown")
    topic_lr[t][ex["lr_routing_choice"]] += 1
    topic_score[t][ex["score_routing_choice"]] += 1

topics = sorted(topic_lr.keys())
lr_lora_pct = [topic_lr[t][1] / sum(topic_lr[t]) for t in topics]
score_lora_pct = [topic_score[t][1] / sum(topic_score[t]) for t in topics]

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = list(range(len(topics)))
ax.bar([p - 0.2 for p in x_pos], score_lora_pct, 0.4, label="ScoreRouting")
ax.bar([p + 0.2 for p in x_pos], lr_lora_pct, 0.4, label="LR-Routing")
ax.set_xticks(x_pos)
ax.set_xticklabels(topics, rotation=45, ha="right")
ax.set_ylabel("Fraction routed to LoRA")
ax.set_title("Per-topic routing: where do the two routers diverge?")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

* Compare against XGBoost routing: run `bash scripts/reproduce_phase0.sh` and load `evaluation/results/phase0/xgboost/feature_importance.tsv` — same feature space, different model, +3.24pp All-acc over LR.
* Inspect specific routing failures by filtering `examples` on `oracle_label != lr_routing_choice` — the LR router gets each of those wrong, and inspecting their text + MRs is a good source of routing-feature ideas.
* The same notebook works on the PERSONAGE routing data: replace `results_dir` with `evaluation/results/appendix_d/` after running `scripts/reproduce_appendix_d.sh`.